# Zonal Statistics Example

This notebook shows an example of how to compute all widget statistics using [exactextract](https://github.com/isciences/exactextract).  
Helper functions live in `src/data_processing/zonal_stats.py`.  
Polygon arrives from the front-end in **EPSG:3857**; each raster is sampled in its own native CRS.

In [ ]:
import json
import sys
from datetime import date, timedelta
from pathlib import Path

sys.path.insert(0, "../src")
from data_processing.zonal_stats import build_frac_dict, build_histogram, frac_range, frac_sum, run_extract
from shapely.geometry import box

In [ ]:
# Raster config — one entry per raster: path + exactextract ops to request.
COG_DIR = Path("../data/processed/rasters/cogs")
SEASONS  = ["1819", "1920", "2021", "2122", "2223", "2324"]

RASTER_CONFIG = {
    "peat_depth":       {"path": COG_DIR / "carbon_peatland"          / "peat_cog.tif",                 "ops": ["mean", "max", "values", "coverage"]},
    "carbon":           {"path": COG_DIR / "carbon_peatland"          / "carbon_cog.tif",               "ops": ["sum", "mean", "values", "coverage"]},
    "inundation_freq":  {"path": COG_DIR / "dynamic_surface_water"    / "inundation_frequency_cog.tif", "ops": ["frac", "unique", "mean"]},
    "inundation_trend": {"path": COG_DIR / "dynamic_surface_water"    / "inundation_trends_cog.tif",    "ops": ["frac", "unique"]},
    "fsi":              {"path": COG_DIR / "flood_susceptibility"      / "flood_susceptibility_cog.tif", "ops": ["frac", "unique", "mean"]},
    "treed_area":       {"path": COG_DIR / "treed_area"               / "treed_area_1984-2022_cog.tif", "ops": ["frac", "unique", "count"]},
    "ecosystem":        {"path": COG_DIR / "ecosystem_classification"  / "ecosystem_classification_cog.tif", "ops": ["frac", "unique"]},
    **{f"snow_end_{s}": {"path": COG_DIR / "snow_dynamics" / f"endL_winter_{s}_cog.tif",    "ops": ["mean"]} for s in SEASONS},
    **{f"snow_len_{s}": {"path": COG_DIR / "snow_dynamics" / f"lengthT_winter_{s}_cog.tif", "ops": ["mean"]} for s in SEASONS},
}

In [ ]:
# Constants
PIXEL_AREA_KM2 = 0.0009  # 30 m × 30 m in km²

SNOW_END_BASE_DATES = {
    "1819": date(2018, 12, 31), "1920": date(2019, 12, 31),
    "2021": date(2020, 12, 31), "2122": date(2021, 12, 31),
    "2223": date(2022, 12, 31), "2324": date(2023, 12, 31),
}

ECOSYSTEM_CLASSES = {
     1: "eco_temperate_perc",  2: "eco_treed_perc",     3: "eco_grassland_perc",
     4: "eco_fire_perc",       5: "eco_graminoid_perc",  6: "eco_shrub_perc",
     7: "eco_emergent_perc",   8: "eco_bog_perc",        9: "eco_mudflats_perc",
    10: "eco_coastal_perc",   11: "eco_marine_perc",    12: "eco_water_perc",
}

In [ ]:
# Sample polygon — ~50 km × 50 km (EPSG:3857).
polygon_3857 = box(-9_720_000, 7_265_000, -9_670_000, 7_315_000)

def ex(key: str) -> dict:
    """Shorthand: run exactextract for a configured raster on the sample polygon."""
    cfg = RASTER_CONFIG[key]
    return run_extract(cfg["path"], polygon_3857, cfg["ops"])

## 1. Peat Depth

In [ ]:
r = ex("peat_depth")

peat_depth = {
    "peat_depth_avg": round(r["mean"], 1),  # cm
    "peat_depth_max": round(r["max"],  1),  # cm
    "chart":          build_histogram(r["values"], r["coverage"]),
}
peat_depth

## 2. Carbon Storage

In [ ]:
r = ex("carbon")

carbon = {
    "carbon_total":   round(r["sum"]  * 9e-7, 6),  # sum [kg/m²] × 900 [m²/pixel] / 1e9 [kg/Mt] → Mt
    "carbon_density": round(r["mean"],        2),   # kg/m²
    "chart":          build_histogram(r["values"], r["coverage"]),
}
carbon

## 3. Inundation Frequency

`0` = permanent land · `1–99` = ephemeral water · `100` = permanent water

In [ ]:
r = ex("inundation_freq")
fracs = build_frac_dict(r["unique"], r["frac"])

inundation_freq = {
    "water_perm_perc":      round(frac_sum(fracs, [100])   * 100, 2),
    "water_ephemeral_perc": round(frac_range(fracs, 1, 99) * 100, 2),
    "land_perm_perc":       round(frac_sum(fracs, [0])     * 100, 2),
    "freq_mean":            round(r["mean"],                       2),
}
inundation_freq

## 4. Inundation Trends

`1–3` = stable (dry/wet/mixed) · `4` = wetter · `5` = drier

In [ ]:
r = ex("inundation_trend")
fracs = build_frac_dict(r["unique"], r["frac"])

inundation_trends = {
    "trend_wetter_perc": round(frac_sum(fracs, [4])       * 100, 2),
    "trend_drier_perc":  round(frac_sum(fracs, [5])       * 100, 2),
    "trend_stable_perc": round(frac_sum(fracs, [1, 2, 3]) * 100, 2),
}
inundation_trends

## 5. Flood Susceptibility Index

`0–30` = low · `31–80` = moderate · `81–100` = high

In [ ]:
r = ex("fsi")
fracs = build_frac_dict(r["unique"], r["frac"])

fsi = {
    "fsi_avg":           round(r["mean"],                     2),
    "fsi_low_perc":      round(frac_range(fracs,  0, 30) * 100, 2),
    "fsi_moderate_perc": round(frac_range(fracs, 31, 80) * 100, 2),
    "fsi_high_perc":     round(frac_range(fracs, 81, 100)* 100, 2),
}
fsi

## 6. End of Snow Period

Pixel values = days elapsed after Dec 31 of the winter-start year.

In [ ]:
snow_end = {
    f"endL_mean_date_{s}": (SNOW_END_BASE_DATES[s] + timedelta(days=round(ex(f"snow_end_{s}")["mean"]))).isoformat()
    for s in SEASONS
}
snow_end

## 7. Total Snow Length

Pixel values = snow-covered days in that winter.

In [ ]:
snow_length = {
    f"lengthT_mean_{s}": round(ex(f"snow_len_{s}")["mean"], 1)
    for s in SEASONS
}
snow_length

## 8. Treed Area (1984–2022)

`0` = never treed · `1` = always treed · `2` = newly treed · `3` = was treed (lost)  
Pixel area = 0.0009 km² (30 m × 30 m)

In [ ]:
r = ex("treed_area")
fracs = build_frac_dict(r["unique"], r["frac"])
n = r["count"]  # coverage-weighted pixel count; area = frac × n × PIXEL_AREA_KM2

f0, f1, f2, f3 = (fracs.get(v, 0.0) for v in range(4))
a0, a1, a2, a3 = (round(f * n * PIXEL_AREA_KM2, 3) for f in [f0, f1, f2, f3])

treed_area = {
    "non_treed_area":     a0,  "always_treed_area":   a1,
    "newly_treed_area":   a2,  "was_treed_area":      a3,
    "total_treed_area":   round(a1 + a2, 3),
    "changed_treed_area": round(a2 - a3, 3),          # +ve = net gain
    "non_treed_perc":     round(f0 * 100, 2),  "always_treed_perc": round(f1 * 100, 2),
    "newly_treed_perc":   round(f2 * 100, 2),  "was_treed_perc":    round(f3 * 100, 2),
}
treed_area

## 9. Ecosystem Classification

Classes 1–12: see `ECOSYSTEM_CLASSES` in the setup cell.

In [ ]:
r = ex("ecosystem")
fracs = build_frac_dict(r["unique"], r["frac"])
dominant = max(fracs, key=fracs.get)

ecosystem = {
    **{name: round(fracs.get(v, 0.0) * 100, 2) for v, name in ECOSYSTEM_CLASSES.items()},
    "ecosystem_count":         len(fracs),
    "dominant_ecosystem":      dominant,
    "dominant_ecosystem_perc": round(fracs[dominant] * 100, 2),
}
ecosystem

## Summary

In [ ]:
stats = {
    **peat_depth, **carbon,
    **inundation_freq, **inundation_trends,
    **fsi,
    **snow_end, **snow_length,
    **treed_area,
    **ecosystem,
}

print(json.dumps(stats, indent=2, default=str))